# DART 공시 원본 수집

그래프 DB의 **지분·계열 관계**에 쓸 공시를 OpenDART 공식 API로 받아 **받은 그대로** 쌓는다. 받을 자료와 이유는 `graphDB.md` 3장을 따른다.

- 이 노트북은 **받기만 한다.** 표를 뽑고 정리하는 일은 `dart_parser.ipynb`가 이 노트북이 쌓은 파일을 읽어서 한다.
- 가공 규칙을 고쳐도 다시 요청할 필요가 없고, 같은 원문에서 나중에 다른 정보(사업의 내용 등)도 뽑을 수 있다.
- 정기보고서 자료는 연도를 정해 두지 않고, **기업마다 가장 최근 보고서**에서 받는다(5절). 결산월이 12월이 아닌 기업, 정정공시, 새로 나온 보고서가 저절로 반영된다.

**받는 자료** (모두 `source/dart/raw/` 아래에 저장)

| 절 | 자료 | OpenDART API | 저장 위치 |
|---|---|---|---|
| 3 | 고유번호 목록 | `corpCode` | `corpCode/corpCode_날짜.zip` |
| 4 | 기업개황 | `company` | `company/종목코드.json` |
| 5 | 정기공시 목록 (최근 2년, 정정 전 보고서 포함) | `list` | `list/종목코드.json` |
| 6 | 타법인 출자현황, 최대주주 현황 (최신 보고서) | `otrCprInvstmntSttus`, `hyslrSttus` | `periodic/API/종목코드_사업연도_보고서코드.json` |
| 7 | 타법인 주식 양수·양도, 합병·분할 결정 | 결정 공시 API 5종 | `decisions/API/종목코드_연도.json` |
| 8 | 사업보고서 원문 (최신 사업보고서) | `document` | `document/종목코드_사업연도_접수번호.zip` |
| - | 요청 기록 | - | `requests_log.csv` |

8절 끝에서 기업·항목마다 고른 보고서를 `source/dart/selected_reports.csv`에 저장한다. `dart_parser`는 이 파일을 보고 어느 파일을 읽을지 정한다.

**실행 전 준비**

- 프로젝트 루트 `.env`에 `OPENDART_API_KEY`가 있어야 한다.
- 대상 기업 목록은 `dart_targets.ipynb`가 만든 `source/targets.csv`에서 읽는다(`stock_code` 열 필수). 파일에 적힌 순서(코스피200 → 코스피 → 코스닥150 → 코스닥 추가)대로 요청하므로, 한도에 걸려 멈춰도 앞쪽 기업부터 받아 둔 상태가 된다. 파일이 없으면 테스트용 6개 기업으로 실행한다.
- 이미 받은 파일은 다시 요청하지 않는다(2절의 규칙). 처음부터 다시 받으려면 `FORCE = True`로 바꾼다.

**걸리는 시간**: 대상 기업 1,067개 기준 처음 실행은 약 18,000건 요청, 2~3시간이다. 하루 한도(약 20,000건) 안에 들어간다. 한도에 걸리면 수집이 멈추고, 다음 날 다시 실행하면 남은 것부터 이어 받는다. 다시 실행하면 정기공시 목록(1,067건)과 올해 수시공시(약 5,300건), 새로 나오거나 정정된 보고서만 받는다.

**한눈에**

| | |
|---|---|
| 처리 순서에서 | `dart_targets` → **`dart_collector`** → `dart_parser` |
| 외부 요청·비용 | 있음. OpenDART API(무료, 하루 약 20,000건 한도). 처음은 약 18,000건·2~3시간, 다시 실행은 목록·새 보고서만 |
| 다시 실행 | 된다. 받은 파일은 다시 요청하지 않는다(`FORCE=True`면 처음부터) |
| 멈추는 경우 | 하루 한도·인증키 오류 같은 응답(`STOP_STATUS`), 네트워크 오류 세 번. 다음 실행 때 남은 것부터 이어 받는다 |

## 1. 설정

In [1]:
# ── 설정: 폴더, 요청 기록, 대상 파일 ──
import csv
import datetime as dt
import json
import re
import time
import xml.etree.ElementTree as ET
import zipfile
from collections import Counter
from pathlib import Path

import pandas as pd
import requests

# 이 노트북은 graph_db 폴더에서 실행한다.
ROOT = Path.cwd()
assert ROOT.name == 'graph_db', 'graph_db 폴더에서 실행해야 한다.'

RAW = ROOT / 'source' / 'dart' / 'raw'          # 받은 원본을 그대로 저장하는 곳
LOG_PATH = RAW / 'requests_log.csv'              # 모든 요청의 기록
TARGETS_FILE = ROOT / 'source' / 'targets.csv'   # 대상 기업 목록
RAW.mkdir(parents=True, exist_ok=True)

# 이번 실행 전까지 쌓인 요청 기록 수. 9절에서 이번 실행의 요청만 골라 볼 때 쓴다.
LOG_ROWS_BEFORE_RUN = len(pd.read_csv(LOG_PATH)) if LOG_PATH.exists() else 0

TODAY = dt.date.today()

# ── 수집 범위 ──────────────────────────────────────────────────────────────
# 정기보고서(사업·반기·분기)는 연도를 정하지 않는다. 기업마다 정기공시 목록에서 가장 최근 보고서를 찾아 받는다(5절).
LIST_LOOKBACK_DAYS = 730        # 정기공시 목록을 검색할 기간(일). 가장 최근 사업보고서가 반드시 들어가도록 2년
LIST_MAX_AGE_DAYS = 7           # 정기공시 목록은 받은 지 이 기간(일)이 지나면 새로 받는다 (새 보고서·정정 확인)

# 수시공시(7절)를 받을 연도: 작년과 올해. 최근 정기보고서 이후의 지분 변화와 회사 구조 변화를 잡는 것이 목적이다.
DECISION_YEARS = [TODAY.year - 1, TODAY.year]

# 기업·항목마다 고른 최신 보고서 기록 (8절 끝에서 저장, dart_parser가 읽는다)
SELECTED_PATH = ROOT / 'source' / 'dart' / 'selected_reports.csv'

# ── 요청 방식 ──────────────────────────────────────────────────────────────
FORCE = False                   # True면 저장된 파일이 있어도 모두 다시 요청한다
SLEEP = 0.2                     # 요청 사이에 쉬는 시간(초)
CORPCODE_MAX_AGE_DAYS = 7       # 고유번호 목록은 이 기간(일)이 지나면 새로 받는다 (신규 상장 반영)

In [2]:
# ── 인증키: .env에서 읽는다(노트북에 쓰지 않는다) ──
def load_env_value(name):
    """graph_db 또는 프로젝트 루트의 .env에서 값을 읽는다."""
    for env_path in [ROOT / '.env', ROOT.parent / '.env']:
        if not env_path.exists():
            continue
        for line in env_path.read_text(encoding='utf-8').splitlines():
            if line.startswith(name + '='):
                return line.split('=', 1)[1].strip().strip('"\'')
    raise RuntimeError(f'.env에서 {name}를 찾지 못했다.')

API_KEY = load_env_value('OPENDART_API_KEY')
print('OpenDART 키 확인:', len(API_KEY), '자리')

OpenDART 키 확인: 40 자리


## 2. 요청과 저장 규칙

**저장 형식**

- JSON 응답은 아래처럼 받은 시각과 요청 값을 함께 감싸서 저장한다. `response` 안은 OpenDART가 준 그대로다. 인증키는 저장하지 않는다.
  ```json
  {"fetched_at": "2026-11-10T14:00:00+09:00", "endpoint": "company", "params": {"corp_code": "00126380"}, "response": {...}}
  ```
- zip 응답(고유번호 목록, 공시 원문)은 받은 파일 그대로 저장한다.
- 정상(`000`)과 자료 없음(`013`)만 저장한다. 그 밖의 오류는 저장하지 않고 `requests_log.csv`에만 남기므로, 다음 실행에서 다시 요청한다.
- 인증키 오류, 하루 한도 초과, 시스템 점검처럼 다음 요청도 모두 실패할 오류가 오면 수집을 바로 멈춘다.

**다시 받는 경우**

저장된 파일이 있으면 요청하지 않는다. 다만 **자료가 아직 바뀔 수 있던 때 받은 파일**은 다시 받는다. 그 기준 날짜(`final_after`)는 자료마다 다르다.

| 자료 | 다시 받는 경우 | 이유 |
|---|---|---|
| 정기공시 목록 | 받은 지 7일이 지나면 | 새 보고서와 정정공시를 찾는다 |
| 정기보고서 API | 그 보고서가 마지막으로 제출·정정된 날(같은 날 포함)까지 받은 파일 | 정정되면 표 내용이 바뀐다 |
| 수시공시 | 그 해 12월 31일까지 받은 파일 | 연도가 끝나기 전에는 새 공시가 계속 나온다 |
| 기업개황, 공시 원문 | 다시 받지 않는다 | 원문은 접수번호마다 내용이 고정이다 |
| 고유번호 목록 | 받은 지 7일이 지나면 | 신규 상장 기업 반영 |

In [3]:
# ── 요청 규칙: 저장하는 응답·멈추는 응답, 인증키 가리기, 요청 기록 ──
BASE_URL = 'https://opendart.fss.or.kr/api/'

SAVE_STATUS = {'000', '013'}        # 저장하는 응답: 000 정상, 013 조회된 자료 없음
STOP_STATUS = {                     # 수집을 멈추는 응답: 이후 요청도 모두 실패한다
    '010': '등록되지 않은 인증키',
    '011': '사용할 수 없는 인증키',
    '012': '접근할 수 없는 IP',
    '020': '하루 요청 한도 초과',
    '800': '시스템 점검 중',
    '901': '계정의 개인정보 보유기간 만료',
}

class DartStop(Exception):
    """더 요청해도 소용없는 오류. 수집을 멈춘다."""


def now_kst():
    """현재 시각을 '2026-11-10T14:00:00+09:00' 형식으로 돌려준다."""
    return dt.datetime.now().astimezone().isoformat(timespec='seconds')


def hide_key(text):
    """오류 메시지에 인증키가 찍히지 않게 가린다."""
    return str(text).replace(API_KEY, '<KEY>')


def write_log(endpoint, params, status, message, saved_to=None):
    """요청 한 건을 requests_log.csv에 한 줄 덧붙인다. 중간에 멈춰도 그때까지의 기록이 남는다."""
    is_new = not LOG_PATH.exists()
    with open(LOG_PATH, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        if is_new:
            writer.writerow(['requested_at', 'endpoint', 'params', 'status', 'message', 'saved_to'])
        writer.writerow([now_kst(), endpoint, json.dumps(params, ensure_ascii=False), status, message,
                         saved_to.relative_to(RAW).as_posix() if saved_to else ''])

In [4]:
# ── 요청 한 번(재시도 포함) ──
def http_get(endpoint, params):
    """OpenDART에 GET 요청을 한 번 보낸다.

    네트워크 오류는 5초, 10초 쉬고 두 번 더 시도한다. 세 번 모두 실패하면 수집을 멈춘다.
    요청이 끝나면 성공·실패와 관계없이 SLEEP초 쉰다.
    """
    for attempt in (1, 2, 3):
        try:
            response = requests.get(BASE_URL + endpoint, params={**params, 'crtfc_key': API_KEY}, timeout=60)
            response.raise_for_status()
            return response
        except requests.RequestException as e:
            if attempt == 3:
                raise DartStop(f'{endpoint} 연결 실패: {hide_key(e)}') from None
            time.sleep(5 * attempt)
        finally:
            time.sleep(SLEEP)


def check_status(endpoint, params, status, message):
    """응답 코드가 저장할 수 있는 것인지 확인한다.

    - 저장할 수 있으면 True
    - 멈춰야 하는 오류면 기록을 남기고 DartStop을 일으킨다
    - 그 밖의 오류면 기록을 남기고 False (다음 실행에서 다시 요청한다)
    """
    if status in SAVE_STATUS:
        return True
    write_log(endpoint, params, status, message)
    if status in STOP_STATUS:
        raise DartStop(f'{status} {STOP_STATUS[status]}: {message}')
    print(f'  ! {endpoint} {params} → {status} {message}')
    return False

In [5]:
# ── 받아서 저장: JSON·zip, 이미 있으면 건너뛰고 요청 기록을 남긴다 ──
def read_saved(path):
    """fetch_json이 저장한 파일을 읽는다."""
    return json.loads(Path(path).read_text(encoding='utf-8'))


def needs_request(path, final_after=None):
    """저장된 파일을 그대로 쓸지, 다시 요청할지 정한다.

    final_after: 이 날짜까지는 자료가 바뀔 수 있다. 이 날짜 이전(같은 날 포함)에 받은 파일은 다시 받는다.
                 None이면 한 번 받은 파일은 다시 받지 않는다.
    """
    if FORCE or not path.exists():
        return True
    if final_after is None:
        return False
    fetched_on = dt.date.fromisoformat(read_saved(path)['fetched_at'][:10])
    return fetched_on <= final_after


def fetch_json(endpoint, params, path, final_after=None, force=False):
    """JSON API 한 건을 받아 path에 저장한다.

    돌려주는 값 (9절과 각 절의 결과 집계에 쓴다)
      'cached'  : 저장된 파일을 그대로 씀 (요청 안 함)
      'saved'   : 새로 받아 저장함
      'no_data' : 자료 없음(013) 응답을 받아 저장함
      'failed'  : 오류라 저장하지 않음. 이전에 저장한 파일이 있으면 그대로 남는다

    force=True면 저장된 파일이 있어도 다시 요청한다 (요청 조건이 바뀐 경우)
    """
    if not force and not needs_request(path, final_after):
        return 'cached'

    body = http_get(endpoint + '.json', params).json()
    status, message = body.get('status', ''), body.get('message', '')
    if not check_status(endpoint, params, status, message):
        return 'failed'

    wrapped = {'fetched_at': now_kst(), 'endpoint': endpoint, 'params': params, 'response': body}
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(wrapped, ensure_ascii=False, indent=1), encoding='utf-8')
    write_log(endpoint, params, status, message, path)
    return 'saved' if status == '000' else 'no_data'


def fetch_zip(endpoint, params, path):
    """zip 파일 한 건을 받아 그대로 저장한다 (고유번호 목록, 공시 원문).

    성공하면 zip 파일이 오고, 실패하면 오류 코드가 담긴 XML이 온다. 돌려주는 값은 fetch_json과 같다.
    """
    if path.exists() and not FORCE:
        return 'cached'

    response = http_get(endpoint + '.xml', params)
    if response.content[:2] == b'PK':          # zip 파일은 항상 'PK'로 시작한다
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(response.content)
        write_log(endpoint, params, '000', 'zip 저장', path)
        return 'saved'

    text = response.content.decode('utf-8', 'replace')
    found = re.search(r'<status>(\d+)</status>.*?<message>(.*?)</message>', text, re.S)
    status, message = (found.group(1), found.group(2)) if found else ('', hide_key(text[:200]))
    check_status(endpoint, params, status, message)
    return 'failed'


def show_result(title, counts):
    """한 절의 결과 집계를 한 줄로 출력한다. 예: 기업개황: 새로 받음 6, 저장된 파일 사용 0"""
    labels = {'saved': '새로 받음', 'no_data': '자료 없음', 'cached': '저장된 파일 사용', 'failed': '실패'}
    print(f'{title}: ' + ', '.join(f'{labels[k]} {counts[k]}' for k in labels if counts[k]))


def show_progress(i, total, every=20):
    """대상 기업이 많을 때 진행 상황을 every개마다 출력한다."""
    if i % every == 0 or i == total:
        print(f'  {i}/{total}')

## 3. 고유번호 목록과 대상 기업

OpenDART의 다른 API는 종목코드가 아니라 **고유번호**(`corp_code`, 8자리)로 기업을 찾는다. 전체 기업의 고유번호 목록(zip)을 받아 대상 기업의 종목코드에 고유번호를 붙인다.

In [6]:
# ── 3. 고유번호 목록(종목코드 → corp_code) ──
CORPCODE_DIR = RAW / 'corpCode'

def latest_corpcode_file():
    """가장 최근에 받은 고유번호 목록 파일과 받은 날짜. 없으면 (None, None)"""
    files = sorted(CORPCODE_DIR.glob('corpCode_*.zip'))     # 파일 이름의 날짜 순서 = 받은 순서
    if not files:
        return None, None
    latest = files[-1]
    return latest, dt.datetime.strptime(latest.stem.split('_')[1], '%Y%m%d').date()


corpcode_file, fetched_on = latest_corpcode_file()
if FORCE or corpcode_file is None or (TODAY - fetched_on).days >= CORPCODE_MAX_AGE_DAYS:
    fetch_zip('corpCode', {}, CORPCODE_DIR / f'corpCode_{TODAY:%Y%m%d}.zip')
    corpcode_file, fetched_on = latest_corpcode_file()
if corpcode_file is None:
    raise RuntimeError('고유번호 목록을 받지 못했다. requests_log.csv를 확인한다.')
print('고유번호 목록:', corpcode_file.name)

# zip 안의 XML: <result><list><corp_code>…</corp_code><corp_name>…</corp_name><stock_code>…</stock_code>…</list>…</result>
with zipfile.ZipFile(corpcode_file) as z:
    xml_root = ET.fromstring(z.read(z.namelist()[0]))
corp_codes = pd.DataFrame([{field.tag: (field.text or '').strip() for field in item}
                           for item in xml_root.iter('list')])
listed = corp_codes[corp_codes['stock_code'] != '']     # 상장 기업만 종목코드가 있다
print('상장 기업', len(listed), '개')

고유번호 목록: corpCode_20261110.zip


상장 기업 4013 개


In [7]:
# ── 3. 대상 기업에 고유번호 붙이기 ──
# 대상 기업 목록 읽기
if TARGETS_FILE.exists():
    targets = pd.read_csv(TARGETS_FILE, dtype={'stock_code': str})
    print('대상 기업 목록:', TARGETS_FILE.relative_to(ROOT))
else:
    targets = pd.DataFrame({
        'stock_code': ['005930', '000660', '373220', '005380', '068270', '247540'],
        'name': ['삼성전자', 'SK하이닉스', 'LG에너지솔루션', '현대차', '셀트리온', '에코프로비엠'],
    })
    print('source/targets.csv가 없어 테스트용 6개 기업으로 실행한다.')
targets['stock_code'] = targets['stock_code'].str.strip().str.zfill(6)

# 종목코드에 고유번호와 DART 회사명을 붙인다
targets = (targets.drop(columns=['corp_code', 'corp_name'], errors='ignore')
                  .merge(listed[['stock_code', 'corp_code', 'corp_name']], on='stock_code', how='left'))
not_found = targets[targets['corp_code'].isna()]
if len(not_found):
    print('고유번호를 찾지 못해 빼는 종목:', not_found['stock_code'].tolist())
targets = targets.dropna(subset=['corp_code']).reset_index(drop=True)
print('대상 기업', len(targets), '개')
targets.head(10)

대상 기업 목록: source/targets.csv
대상 기업 1067 개


,stock_code,name,market,priority,group,wi26_sector,kospi200,kosdaq150,kosdaq_global,section,mktcap_avg_억,trdval_avg_억,corp_code,corp_name
0,005930,삼성전자,KOSPI,1,코스피200,반도체,True,False,False,NaN,12724718.0,42882.1,00126380,삼성전자
1,000660,SK하이닉스,KOSPI,1,코스피200,반도체,True,False,False,NaN,9691807.0,54344.9,00164779,SK하이닉스
2,402340,SK스퀘어,KOSPI,1,코스피200,반도체,True,False,False,NaN,1085405.0,6004.0,01596425,SK스퀘어
3,373220,LG에너지솔루션,KOSPI,1,코스피200,IT가전,True,False,False,NaN,886977.0,1927.9,01515323,LG에너지솔루션
4,009150,삼성전기,KOSPI,1,코스피200,IT하드웨어,True,False,False,NaN,860173.0,9899.3,00126371,삼성전기
5,207940,삼성바이오로직스,KOSPI,1,코스피200,건강관리,True,False,False,NaN,647171.0,1194.1,00877059,삼성바이오로직스
6,005380,현대차,KOSPI,1,코스피200,자동차,True,False,False,NaN,602261.0,1778.0,00164742,현대자동차
7,105560,KB금융,KOSPI,1,코스피200,은행,True,False,False,NaN,551539.0,1489.5,00688996,KB금융
8,032830,삼성생명,KOSPI,1,코스피200,보험,True,False,False,NaN,501900.0,738.6,00126256,삼성생명
9,028260,삼성물산,KOSPI,1,코스피200,"상사,자본재",True,False,False,NaN,477867.0,1134.5,00149655,삼성물산


## 4. 기업개황

영문 정식명칭, 법인구분, 업종코드, 홈페이지 등. 기업 노드와 영문 검색어에 쓴다.

In [8]:
# ── 4. 기업개황 ──
counts = Counter()
for i, t in enumerate(targets.itertuples(), start=1):
    counts[fetch_json('company', {'corp_code': t.corp_code}, RAW / 'company' / f'{t.stock_code}.json')] += 1
    show_progress(i, len(targets))
show_result('기업개황', counts)

  20/1067
  40/1067
  60/1067
  80/1067
  100/1067
  120/1067
  140/1067
  160/1067
  180/1067
  200/1067
  220/1067
  240/1067
  260/1067
  280/1067
  300/1067
  320/1067
  340/1067
  360/1067
  380/1067
  400/1067
  420/1067
  440/1067
  460/1067
  480/1067
  500/1067
  520/1067
  540/1067
  560/1067
  580/1067
  600/1067
  620/1067
  640/1067
  660/1067
  680/1067
  700/1067
  720/1067
  740/1067
  760/1067
  780/1067
  800/1067
  820/1067
  840/1067
  860/1067
  880/1067
  900/1067
  920/1067
  940/1067
  960/1067
  980/1067
  1000/1067
  1020/1067
  1040/1067
  1060/1067
  1067/1067
기업개황: 저장된 파일 사용 1067


## 5. 정기공시 목록과 최신 보고서 고르기

OpenDART의 정기보고서 API는 **사업연도와 보고서 코드를 지정한 그 보고서의 표만** 돌려준다. 최신 보고서를 알아서 골라 주지 않으므로, 기업마다 정기공시 목록을 받아 가장 최근 보고서를 직접 고른다.

1. 공시검색(`list`)으로 최근 2년의 정기공시(사업·반기·분기보고서)를 정정 전 보고서까지 모두 받는다(`pblntf_ty=A`, `last_reprt_at=N`).
2. 원본과 정정본(`[기재정정]`, `[첨부정정]`, `[첨부추가]`)은 같은 보고서로 묶는다. 보고서명에서 `[ ]`를 뺀 이름이 같으면 같은 보고서다.
3. 보고서명 괄호 안의 기간으로 사업연도와 보고서 코드를 정한다.

| 보고서명 예 | 사업연도 | 보고서 코드 |
|---|---|---|
| `사업보고서 (2025.12)` | 2025 | `11011` |
| `반기보고서 (2026.06)` | 2026 | `11012` |
| `분기보고서 (2026.03)` (12월 결산) | 2026 | `11013` 1분기 |
| `분기보고서 (2026.09)` (12월 결산) | 2026 | `11014` 3분기 |
| `사업보고서 (2026.03)` (3월 결산) | 2026 | `11011` |
| `반기보고서 (2026.09)` (3월 결산) | 2026 | `11012` |

- 사업연도는 **보고서 기간이 끝나는 해**다. 결산월이 12월이 아닌 기업도 같다(2026-11-11 받은 자료로 확인).
- 분기보고서는 결산월에서 3개월째면 1분기, 9개월째면 3분기다. 결산월은 **DART에 등록된 결산월(기업개황 `acc_mt`)**을 쓴다. 실제 결산월과 다르더라도 DART가 이 결산월로 분기 코드를 매기기 때문이다. 예: MH머트리얼즈는 실제 6월 결산이지만 기업개황은 12월이고, `분기보고서 (2026.03)`을 1분기(`11013`)로 돌려준다(2026-11-11 확인).

In [9]:
# ── 5. 정기공시 목록과 항목마다 최신 보고서 고르기 ──
REPORT_NAME_RE = re.compile(r'^(?:\[[^\]]+\])?(?P<kind>사업보고서|반기보고서|분기보고서) \((?P<period>\d{4}\.\d{2})\)')


def group_reports(list_file):
    """정기공시 목록을 보고서별로 묶는다.

    돌려주는 값: {('반기보고서', '2026.06'): [목록 항목, ...]}  (항목은 접수번호가 큰(최신) 순)
    같은 보고서의 원본과 정정본은 보고서명 앞의 [기재정정] 같은 표시만 다르므로 한 묶음이 된다.
    """
    groups = {}
    for item in read_saved(list_file)['response'].get('list', []):
        found = REPORT_NAME_RE.match(item['report_nm'])
        if found:
            groups.setdefault((found['kind'], found['period']), []).append(item)
    return {key: sorted(items, key=lambda e: e['rcept_no'], reverse=True) for key, items in groups.items()}


def latest_report(groups, kinds):
    """kinds(보고서 종류 목록)에 속하는 보고서 중 기간이 가장 최근인 것의 키. 없으면 None

    기간은 'YYYY.MM' 문자열이라 문자열 크기 비교가 곧 날짜 비교다.
    """
    keys = [key for key in groups if key[0] in kinds]
    return max(keys, key=lambda key: key[1]) if keys else None


def reprt_code_of(kind, period, fy_end_month):
    """보고서 종류와 기간(YYYY.MM)으로 OpenDART 보고서 코드를 정한다. 알 수 없으면 None

    분기보고서는 결산월에서 몇 달 지난 시점인지로 1분기(3개월)와 3분기(9개월)를 가른다.
    """
    if kind == '사업보고서':
        return '11011'
    if kind == '반기보고서':
        return '11012'
    months_after_fy_end = (int(period[5:]) - fy_end_month) % 12
    return {3: '11013', 9: '11014'}.get(months_after_fy_end)


list_counts = Counter()
reports = {}        # 종목코드 → {'groups': 보고서 묶음, 'fy_end_month': DART에 등록된 결산월}
for i, t in enumerate(targets.itertuples(), start=1):
    params = {'corp_code': t.corp_code,
              'bgn_de': (TODAY - dt.timedelta(days=LIST_LOOKBACK_DAYS)).strftime('%Y%m%d'),
              'end_de': TODAY.strftime('%Y%m%d'),
              'pblntf_ty': 'A', 'last_reprt_at': 'N', 'page_count': '100'}
    list_file = RAW / 'list' / f'{t.stock_code}.json'
    list_counts[fetch_json('list', params, list_file,
                           final_after=TODAY - dt.timedelta(days=LIST_MAX_AGE_DAYS))] += 1
    if not list_file.exists():
        continue
    if int(read_saved(list_file)['response'].get('total_page') or 1) > 1:
        print(f'  ! {t.corp_name}: 목록이 한 페이지(100건)를 넘어 일부가 빠졌다. LIST_LOOKBACK_DAYS를 줄인다.')

    # 분기 코드는 DART에 등록된 결산월 기준으로 매겨진다 (실제 결산월과 다른 기업이 있어도 이 값을 쓴다)
    fy_end_month = int(read_saved(RAW / 'company' / f'{t.stock_code}.json')['response']['acc_mt'])
    reports[t.stock_code] = {'groups': group_reports(list_file), 'fy_end_month': fy_end_month}
    show_progress(i, len(targets), every=100)

show_result('정기공시 목록', list_counts)
non_dec = {t.corp_name: reports[t.stock_code]['fy_end_month'] for t in targets.itertuples()
           if t.stock_code in reports and reports[t.stock_code]['fy_end_month'] != 12}
print('결산월이 12월이 아닌 기업:', non_dec or '없음')

  100/1067
  200/1067
  300/1067


  400/1067


  500/1067


  600/1067
  700/1067
  800/1067


  900/1067


  1000/1067


  1067/1067
정기공시 목록: 저장된 파일 사용 1067
결산월이 12월이 아닌 기업: {'신영증권': 3, '현대약품': 11, '방림': 9, '세원정공': 6, '동원모빌리티': 3, '기신정기': 3, '오리엔트바이오': 3, '금비': 9, '형지엘리트': 6, '한스바이오메드': 9}


## 6. 정기보고서: 타법인 출자현황, 최대주주 현황

기업마다 **그 표가 들어 있는 보고서 중 가장 최근 것** 하나를 받는다.

| 표 | 들어 있는 보고서 | 예 (2026-11 기준 12월 결산) |
|---|---|---|
| 타법인 출자현황 | 사업·반기보고서 (분기보고서에서는 생략된다) | 반기보고서 (2026.06) |
| 최대주주 현황 | 사업·반기·분기보고서 | 3분기보고서가 나왔으면 분기보고서 (2026.09), 아니면 반기보고서 (2026.06) |

- 응답 안의 접수번호(`rcept_no`)가 고른 보고서(원본 또는 정정본)의 것인지 9절에서 확인한다.
- 보고서가 정정되면 마지막 제출일이 바뀌므로 다시 받는다(2절).
- 예전에 받은 이전 보고서의 파일은 지우지 않는다. 어느 파일이 최신인지는 `selected_reports.csv`에 남긴다.

In [10]:
# ── 6. 정기보고서 표: 타법인 출자현황, 최대주주 현황 ──
PERIODIC_APIS = {   # API: (표 이름, 이 표가 들어 있는 보고서 종류)
    'otrCprInvstmntSttus': ('타법인 출자현황', ['사업보고서', '반기보고서']),
    'hyslrSttus': ('최대주주 현황', ['사업보고서', '반기보고서', '분기보고서']),
}

selected = []       # 기업·항목마다 고른 보고서. 8절 끝에서 selected_reports.csv로 저장한다


def select(t, item, key, versions, bsns_year, reprt_code, file):
    """고른 보고서를 selected에 기록한다."""
    selected.append({
        'stock_code': t.stock_code, 'corp_name': t.corp_name, 'item': item,
        'report': f'{key[0]} ({key[1]})', 'bsns_year': bsns_year, 'reprt_code': reprt_code,
        'latest_report_nm': versions[0]['report_nm'], 'latest_rcept_no': versions[0]['rcept_no'],
        'all_rcept_no': ';'.join(v['rcept_no'] for v in versions),
        'file': file.relative_to(RAW).as_posix() if file else '',
    })


counts = Counter()
for i, t in enumerate(targets.itertuples(), start=1):
    info = reports.get(t.stock_code)
    if info is None:
        continue
    for api, (label, kinds) in PERIODIC_APIS.items():
        key = latest_report(info['groups'], kinds)
        if key is None:
            continue
        versions = info['groups'][key]
        reprt_code = reprt_code_of(key[0], key[1], info['fy_end_month'])
        if reprt_code is None:
            print(f'  ! {t.corp_name} {versions[0]["report_nm"]}: 몇 분기인지 알 수 없어 건너뛴다.')
            continue
        bsns_year = key[1][:4]
        path = RAW / 'periodic' / api / f'{t.stock_code}_{bsns_year}_{reprt_code}.json'
        params = {'corp_code': t.corp_code, 'bsns_year': bsns_year, 'reprt_code': reprt_code}
        # 보고서가 마지막으로 제출·정정된 날까지 받은 파일은 다시 받는다
        last_filed = max(dt.datetime.strptime(v['rcept_dt'], '%Y%m%d').date() for v in versions)
        counts[fetch_json(api, params, path, final_after=last_filed)] += 1
        select(t, label, key, versions, bsns_year, reprt_code, path)
    show_progress(i, len(targets), every=100)

show_result('정기보고서', counts)
if selected:    # 고른 보고서가 많은 순서 (예: 최대주주 현황 / 분기보고서 (2026.09) 500)
    print(pd.DataFrame(selected).groupby(['item', 'report']).size().sort_values(ascending=False).head(10).to_string())

  100/1067
  200/1067


  300/1067


  400/1067
  500/1067


  600/1067


  700/1067
  800/1067
  900/1067


  1000/1067


  1067/1067
정기보고서: 새로 받음 9, 저장된 파일 사용 2119
item      report         
타법인 출자현황  반기보고서 (2026.06)    1052
최대주주 현황   반기보고서 (2026.06)    1036
          분기보고서 (2026.09)      17
          분기보고서 (2026.06)       6
타법인 출자현황  반기보고서 (2026.03)       3
          사업보고서 (2026.03)       3
          사업보고서 (2026.06)       3
최대주주 현황   사업보고서 (2026.06)       2
          반기보고서 (2026.09)       1
          분기보고서 (2026.03)       1


## 7. 수시공시: 타법인 주식 양수·양도, 합병·분할 결정

정기보고서 사이에 생긴 지분 변화와 회사 구조 변화를 잡는다. 연도별로 나눠 받는다. 지난 연도는 한 번 받으면 끝이고, 올해는 실행할 때마다 다시 받는다.

In [11]:
# ── 7. 수시공시: 양수·양도, 합병·분할 결정 ──
DECISION_APIS = {
    'otcprStkInvscrInhDecsn': '타법인 주식 및 출자증권 양수결정',
    'otcprStkInvscrTrfDecsn': '타법인 주식 및 출자증권 양도결정',
    'cmpMgDecsn': '회사합병 결정',
    'cmpDvDecsn': '회사분할 결정',
    'cmpDvmgDecsn': '회사분할합병 결정',
}

counts = Counter()
for i, t in enumerate(targets.itertuples(), start=1):
    for year in DECISION_YEARS:
        if year > TODAY.year:                       # 아직 오지 않은 연도는 건너뛴다
            continue
        period_end = min(dt.date(year, 12, 31), TODAY)
        params = {'corp_code': t.corp_code, 'bgn_de': f'{year}0101', 'end_de': period_end.strftime('%Y%m%d')}
        for api in DECISION_APIS:
            path = RAW / 'decisions' / api / f'{t.stock_code}_{year}.json'
            counts[fetch_json(api, params, path, final_after=dt.date(year, 12, 31))] += 1
    show_progress(i, len(targets))
show_result('수시공시', counts)

  20/1067


  40/1067


  60/1067


  80/1067


  100/1067


  120/1067


... (44 outputs left out)

  1020/1067


  1040/1067


  1060/1067


  1067/1067
수시공시: 새로 받음 86, 자료 없음 5249, 저장된 파일 사용 5335


## 8. 사업보고서 원문

종속회사 현황(XII. 상세표)과 관계기업·공동기업 투자(연결재무제표 주석)는 API 항목이 없어 원문에서 뽑아야 한다. **가장 최근 사업보고서**의 원문을 받는다.

1. 5절의 정기공시 목록에서 가장 최근 사업보고서를 고른다.
2. 원문 후보를 접수번호가 큰(최신) 것부터 고른다. `[첨부정정]`은 감사보고서 같은 첨부서류만 다시 낸 공시라 본문이 없으므로 후보에서 뺀다.
3. 후보의 원문 zip(`document`)을 받고, **zip 안에 사업보고서 본문이 있는지 확인한다.** 없으면(파일 없음 `014`, 첨부서류만 있음) 그 이전 보고서를 받는다.

| 가장 최근 보고서 | 원문 zip 내용 | 처리 |
|---|---|---|
| 원본, `[기재정정]` | 고친 내용까지 반영된 사업보고서 전체 | 그대로 쓴다 |
| `[첨부추가]` | 사업보고서 본문 + 추가된 첨부서류 | 그대로 쓴다 |
| `[첨부정정]` | 파일 없음(`014`) 또는 감사보고서만 | 후보에서 빼고 이전 보고서를 받는다 |

- 반기·분기보고서의 재무제표 주석은 요약본이라, 원문은 사업보고서에서만 받는다.
- 나중에 정정공시나 새 사업보고서가 나오면 접수번호가 달라 새 zip을 받는다. 이전 zip은 지우지 않는다. 어느 zip을 쓸지는 `selected_reports.csv`에 남긴다.

In [12]:
# ── 8. 사업보고서 원문(zip): 본문이 든 보고서를 고른다 ──
def has_business_report(zip_path):
    """원문 zip에 사업보고서 본문이 들어 있는지 확인한다.

    zip 안의 XML 파일마다 앞부분에 <DOCUMENT-NAME>이 있다. 본문은 '사업보고서',
    첨부서류는 '감사보고서', '연결감사보고서' 등이다.
    """
    with zipfile.ZipFile(zip_path) as z:
        for name in z.namelist():
            with z.open(name) as f:
                head = f.read(5000).decode('utf-8', 'ignore')
            found = re.search(r'<DOCUMENT-NAME[^>]*>(.*?)</DOCUMENT-NAME>', head)
            if found and found.group(1).strip() == '사업보고서':
                return True
    return False


doc_counts = Counter()
no_report, no_body = [], []           # 사업보고서가 없는 기업, 본문을 받지 못한 기업
for i, t in enumerate(targets.itertuples(), start=1):
    info = reports.get(t.stock_code)
    key = latest_report(info['groups'], ['사업보고서']) if info else None
    if key is None:
        no_report.append(t.corp_name)
        continue
    versions = info['groups'][key]
    bsns_year = key[1][:4]
    candidates = [v for v in versions if not v['report_nm'].startswith('[첨부정정]')]

    # 최신 후보부터 받아서 사업보고서 본문이 있으면 멈춘다
    chosen = None
    for report in candidates:
        doc_path = RAW / 'document' / f'{t.stock_code}_{bsns_year}_{report["rcept_no"]}.zip'
        doc_counts[fetch_zip('document', {'rcept_no': report['rcept_no']}, doc_path)] += 1
        if doc_path.exists() and has_business_report(doc_path):
            chosen = doc_path
            break
        print(f'  - {t.corp_name} {report["report_nm"]} {report["rcept_no"]}: 본문이 없어 이전 보고서를 받는다.')
    if chosen is None:
        no_body.append(t.corp_name)
    select(t, '사업보고서 원문', key, versions, bsns_year, '11011', chosen)
    show_progress(i, len(targets), every=100)

show_result('사업보고서 원문', doc_counts)
print('사업보고서가 없는 기업:', no_report or '없음')
print('본문을 받지 못한 기업:', no_body or '없음')

  ! document {'rcept_no': '20260619000667'} → 014 파일이 존재하지 않습니다.
  - KB금융 [기재정정]사업보고서 (2025.12) 20260619000667: 본문이 없어 이전 보고서를 받는다.
  100/1067
  200/1067
  300/1067


  400/1067


  ! document {'rcept_no': '20260619000568'} → 014 파일이 존재하지 않습니다.
  - DH오토넥스 [기재정정]사업보고서 (2025.12) 20260619000568: 본문이 없어 이전 보고서를 받는다.
  500/1067
  600/1067
  700/1067
  800/1067


  900/1067
  1000/1067
  1067/1067
사업보고서 원문: 저장된 파일 사용 1063, 실패 2
사업보고서가 없는 기업: ['한화머시너리앤서비스홀딩스', '엘리스', '마키나락스', '디티에스']
본문을 받지 못한 기업: 없음


In [13]:
# ── 고른 보고서 기록(selected_reports.csv): dart_parser가 이 파일로 읽을 파일을 정한다 ──
# 고른 보고서 기록: dart_parser는 이 파일의 file 열을 읽는다 (file이 비어 있으면 받지 못한 것)
#   item: 타법인 출자현황 / 최대주주 현황 / 사업보고서 원문
#   report: 고른 보고서 (예: 반기보고서 (2026.06)), latest_report_nm·latest_rcept_no: 그 보고서의 최신 정정본
selected_df = pd.DataFrame(selected)
selected_df.to_csv(SELECTED_PATH, index=False, encoding='utf-8-sig')
print('저장:', SELECTED_PATH.relative_to(ROOT), len(selected_df), '줄')
selected_df.head()

저장: source/dart/selected_reports.csv 3191 줄


,stock_code,corp_name,item,report,bsns_year,reprt_code,latest_report_nm,latest_rcept_no,all_rcept_no,file
0,005930,삼성전자,타법인 출자현황,반기보고서 (2026.06),2026,11012,반기보고서 (2026.06),20260814003699,20260814003699,periodic/otrCprInvstmntSttus/005930_2026_11012...
1,005930,삼성전자,최대주주 현황,반기보고서 (2026.06),2026,11012,반기보고서 (2026.06),20260814003699,20260814003699,periodic/hyslrSttus/005930_2026_11012.json
2,000660,SK하이닉스,타법인 출자현황,반기보고서 (2026.06),2026,11012,반기보고서 (2026.06),20260814003509,20260814003509,periodic/otrCprInvstmntSttus/000660_2026_11012...
3,000660,SK하이닉스,최대주주 현황,반기보고서 (2026.06),2026,11012,반기보고서 (2026.06),20260814003509,20260814003509,periodic/hyslrSttus/000660_2026_11012.json
4,402340,SK스퀘어,타법인 출자현황,반기보고서 (2026.06),2026,11012,[기재정정]반기보고서 (2026.06),20260814003471,20260814003471;20260814002738;20260814002363,periodic/otrCprInvstmntSttus/402340_2026_11012...


## 9. 수집 결과 확인

- 첫 번째 표: 기업마다 고른 보고서와 받은 표의 줄 수. 표 위에 항목별로 빠진 기업 수와, 받은 표의 접수번호가 고른 보고서와 다른 경우를 출력한다.
- 두 번째 표: 이번 실행에서 실패한 요청. 비어 있으면 모두 정상이다. 실패한 요청은 다시 실행하면 다시 시도한다.

In [14]:
# ── 9. 확인: 기업마다 고른 보고서와 받은 표의 줄 수 ──
def response_status(path):
    """저장된 JSON의 응답 코드 ('000' 또는 '013'). 파일이 없으면 ''"""
    return read_saved(path)['response'].get('status', '') if path.exists() else ''


def item_count(path):
    """저장된 JSON 응답의 list 항목 수. 파일이 없으면 None"""
    return len(read_saved(path)['response'].get('list', [])) if path.exists() else None


chosen_by_company = {}
for s in selected:
    chosen_by_company.setdefault(s['stock_code'], {})[s['item']] = s

rows, wrong_report = [], []
for t in targets.itertuples():
    chosen = chosen_by_company.get(t.stock_code, {})
    row = {'stock_code': t.stock_code, 'corp_name': t.corp_name,
           '기업개황': response_status(RAW / 'company' / f'{t.stock_code}.json')}
    for label, _ in PERIODIC_APIS.values():
        s = chosen.get(label)
        row[label] = s['report'] if s else ''
        row[f'{label} 줄 수'] = item_count(RAW / s['file']) if s else None
        # 받은 표의 접수번호가 고른 보고서(원본·정정본 중 하나)의 것인지 확인
        if s and (RAW / s['file']).exists():
            got = {x['rcept_no'] for x in read_saved(RAW / s['file'])['response'].get('list', [])}
            if not got <= set(s['all_rcept_no'].split(';')):
                wrong_report.append((t.corp_name, label, s['report'], sorted(got)))
    row['수시공시'] = sum(item_count(p) or 0 for p in (RAW / 'decisions').glob(f'*/{t.stock_code}_*.json'))
    s = chosen.get('사업보고서 원문')
    row['사업보고서 원문'] = f"{s['report']} {Path(s['file']).stem.split('_')[-1]}" if s and s['file'] else ''
    rows.append(row)

status = pd.DataFrame(rows)
missing = {col: int((status[col] == '').sum()) for col in ['기업개황', '사업보고서 원문'] + [l for l, _ in PERIODIC_APIS.values()]}
print('받지 못한 기업 수:', {k: v for k, v in missing.items() if v} or '없음')
print('고른 보고서와 접수번호가 다른 표:', wrong_report or '없음')
status

받지 못한 기업 수: {'사업보고서 원문': 4, '타법인 출자현황': 3, '최대주주 현황': 3}
고른 보고서와 접수번호가 다른 표: 없음


,stock_code,corp_name,기업개황,타법인 출자현황,타법인 출자현황 줄 수,최대주주 현황,최대주주 현황 줄 수,수시공시,사업보고서 원문
0,005930,삼성전자,000,반기보고서 (2026.06),128.0,반기보고서 (2026.06),25.0,0,사업보고서 (2025.12) 20260310002820
1,000660,SK하이닉스,000,반기보고서 (2026.06),45.0,반기보고서 (2026.06),13.0,0,사업보고서 (2025.12) 20260317000635
2,402340,SK스퀘어,000,반기보고서 (2026.06),17.0,반기보고서 (2026.06),14.0,0,사업보고서 (2025.12) 20260317000830
3,373220,LG에너지솔루션,000,반기보고서 (2026.06),41.0,반기보고서 (2026.06),2.0,0,사업보고서 (2025.12) 20260312000217
4,009150,삼성전기,000,반기보고서 (2026.06),23.0,반기보고서 (2026.06),8.0,0,사업보고서 (2025.12) 20260310003071
...,...,...,...,...,...,...,...,...,...
1062,200710,에이디테크놀로지,000,반기보고서 (2026.06),11.0,반기보고서 (2026.06),3.0,0,사업보고서 (2025.12) 20260320001390
1063,265520,AP시스템,000,반기보고서 (2026.06),9.0,반기보고서 (2026.06),6.0,1,사업보고서 (2025.12) 20260319001249
1064,042520,한스바이오메드,000,반기보고서 (2026.03),16.0,분기보고서 (2026.06),6.0,0,사업보고서 (2025.09) 20251218000494
1065,115450,HLB테라퓨틱스,000,반기보고서 (2026.06),13.0,반기보고서 (2026.06),12.0,0,사업보고서 (2025.12) 20260319001264


In [15]:
# ── 확인: 이번 실행의 요청 수와 응답 상태 ──
log = pd.read_csv(LOG_PATH, dtype=str).fillna('') if LOG_PATH.exists() else pd.DataFrame()
this_run = log.iloc[LOG_ROWS_BEFORE_RUN:]      # 이번 실행에서 덧붙은 줄만
print('이번 실행 요청', len(this_run), '건')

failed = this_run[~this_run['status'].isin(SAVE_STATUS)] if len(this_run) else this_run   # 저장하지 못한 요청
failed

이번 실행 요청 5346 건


,requested_at,endpoint,params,status,message,saved_to
36367,2026-11-11T16:10:02+09:00,document,"{""rcept_no"": ""20260619000667""}",014,파일이 존재하지 않습니다.,
36368,2026-11-11T16:10:03+09:00,document,"{""rcept_no"": ""20260619000568""}",014,파일이 존재하지 않습니다.,
